# Notebook 02 — Confidence & Review

Runs extraction three times per document and checks whether the passes agree. Confidence is not the model's self-reported certainty — it's measured: does an independent re-extraction find the same covenant, with the same threshold, at the same citation? Combined with schema validation (does the record have everything it structurally needs?), this produces a genuine confidence signal. Anything that doesn't meet the bar is flagged for human review rather than accepted automatically.

**Input:** `synthetic_agreements/*.txt`
**Output:** `records/02_confidence_output.json`

In [12]:
import json
import os
from pathlib import Path

import anthropic
from dotenv import load_dotenv

load_dotenv()  # searches this notebook's directory and upward for a .env file

assert os.environ.get("ANTHROPIC_API_KEY"), (
    "ANTHROPIC_API_KEY not found. Check that .env exists in the project root "
    "and contains a line like ANTHROPIC_API_KEY=sk-ant-..."
)

client = anthropic.Anthropic()  # picks up ANTHROPIC_API_KEY from the environment automatically

AGREEMENTS_DIR = Path("../synthetic_agreements")
RECORDS_DIR = Path("../records")
RECORDS_DIR.mkdir(exist_ok=True)

print("Setup complete. Client initialized, directories ready.")

Setup complete. Client initialized, directories ready.


## Extraction schema (same as Notebook 01)

Reused unchanged, since we need to run extraction multiple times against the identical schema — confidence is meaningless if the passes aren't even asking the same question. This notebook stays standalone rather than importing from Notebook 01, so the schema is redeclared here in full.

In [13]:
COVENANT_EXTRACTION_TOOL = {
    "name": "record_extracted_covenants",
    "description": "Record every financial covenant found in the source document, with every value grounded in an exact source citation.",
    "input_schema": {
        "type": "object",
        "properties": {
            "covenants": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "covenant_name": {
                            "type": "string",
                            "description": "The defined term for this covenant exactly as it appears in the document, e.g. 'Delinquency Ratio' or 'Consolidated Net Leverage Ratio'."
                        },
                        "definition_text": {
                            "type": ["string", "null"],
                            "description": "A concise paraphrase of how this metric is calculated (numerator and denominator), based only on the document's own definition. Null if no governing definition could be located."
                        },
                        "definition_citation": {
                            "type": ["string", "null"],
                            "description": "The exact, verbatim substring from the source document that defines this term. Must be copied exactly, not paraphrased. Null if not found."
                        },
                        "threshold_operator": {
                            "type": ["string", "null"],
                            "description": "The compliance direction, e.g. 'must not exceed' or 'must be greater than or equal to'."
                        },
                        "threshold_tiers": {
                            "type": "array",
                            "description": "One entry per distinct threshold value and the period range it governs. Most covenants have exactly one tier. A covenant amended to change the threshold over time will have more than one.",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "applies_from": {
                                        "type": ["string", "null"],
                                        "description": "Description or date of the earliest period this tier's threshold governs, as stated in the document."
                                    },
                                    "applies_to": {
                                        "type": ["string", "null"],
                                        "description": "Description or date of the last period this tier's threshold governs, or null if open-ended / still current."
                                    },
                                    "threshold_value": {
                                        "type": "string",
                                        "description": "The exact numeric threshold, e.g. '5.00%' or '3.50:1.00'."
                                    },
                                    "citation": {
                                        "type": "string",
                                        "description": "The exact, verbatim substring from the source document stating this threshold value."
                                    }
                                },
                                "required": ["threshold_value", "citation"]
                            }
                        },
                        "supersedes": {
                            "type": ["string", "null"],
                            "description": "If this document amends a covenant from a prior document, a description of what section/covenant it modifies. Null otherwise."
                        },
                        "document_effective_date": {
                            "type": ["string", "null"],
                            "description": "The date this document (or amendment) itself became effective, as stated in the document."
                        },
                        "extraction_notes": {
                            "type": ["string", "null"],
                            "description": "Any ambiguity, inconsistency, or uncertainty worth flagging. Null if none."
                        }
                    },
                    "required": ["covenant_name", "threshold_tiers"]
                }
            }
        },
        "required": ["covenants"]
    }
}

SYSTEM_PROMPT = """You are extracting financial covenant terms from a loan or credit agreement for a portfolio monitoring system. Follow these rules strictly:

1. Extract only. Never infer, estimate, or generate a value that is not explicitly stated in the source text.
2. Every citation field must be an exact, verbatim substring copied from the source document -- not a paraphrase, not a summary, not a reconstruction. If you cannot find an exact matching span, leave the citation null.
3. If a field's value cannot be verified against explicit text in the document, return null for that field rather than guessing.
4. Extract every financial covenant in the document -- a document may contain more than one.
5. Pay close attention to cross-references: a covenant's operative clause and its governing definition are often in different sections. Resolve the full definition before extracting the threshold.
6. If the document is an amendment, extract the covenant as amended, and populate 'supersedes' and 'document_effective_date' accordingly."""

## Extraction and citation verification

Both functions carried over from Notebook 01 in their final, hardened form — `extract_covenants` includes the normalizer built after debugging four different response-shape issues last time, and `verify_citations` independently checks every citation against the actual source text rather than trusting the model's self-report.

In [14]:
def extract_covenants(document_text: str, document_name: str) -> list[dict]:
    response = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=4000,
        system=SYSTEM_PROMPT,
        tools=[COVENANT_EXTRACTION_TOOL],
        tool_choice={"type": "tool", "name": "record_extracted_covenants"},
        messages=[
            {"role": "user", "content": f"Source document: {document_name}\n\n{document_text}"}
        ],
    )
    for block in response.content:
        if block.type == "tool_use":
            covenants = block.input["covenants"]
            while isinstance(covenants, str):
                covenants = json.loads(covenants)
            if isinstance(covenants, dict):
                covenants = covenants.get("covenants", [covenants])
            return covenants
    return []


def verify_citations(covenants: list[dict], source_text: str) -> list[dict]:
    for covenant in covenants:
        def_citation = covenant.get("definition_citation")
        covenant["definition_citation_verified"] = (def_citation in source_text) if def_citation else None
        for tier in covenant.get("threshold_tiers", []):
            tier["citation_verified"] = tier["citation"] in source_text
    return covenants

## Run three extraction passes per document

Each document is extracted three separate times, independently. Agreement across these passes — not the model's self-reported confidence — is the actual signal. Raw results are saved to disk immediately after this cell, before any alignment or scoring logic runs, so a bug downstream doesn't cost you the API spend already made here.

In [15]:
N_PASSES = 3

documents = {
    "facility_a_credit_agreement.txt": "Facility A",
    "facility_b_credit_agreement.txt": "Facility B",
    "facility_c_credit_agreement.txt": "Facility C (original)",
    "facility_c_amendment_1.txt": "Facility C (Amendment 1)",
}

all_passes = {}

for filename, label in documents.items():
    text = (AGREEMENTS_DIR / filename).read_text()
    print(f"Running {N_PASSES} extraction passes on {label} ({filename})...")
    passes = []
    for i in range(N_PASSES):
        covenants = extract_covenants(text, filename)
        covenants = verify_citations(covenants, text)
        passes.append(covenants)
        print(f"  Pass {i+1}: {len(covenants)} covenant(s)")
    all_passes[filename] = passes

raw_passes_path = RECORDS_DIR / "02_raw_passes.json"
raw_passes_path.write_text(json.dumps(all_passes, indent=2))
print(f"\nAll passes complete. Raw output saved to {raw_passes_path}")

Running 3 extraction passes on Facility A (facility_a_credit_agreement.txt)...
  Pass 1: 2 covenant(s)
  Pass 2: 2 covenant(s)
  Pass 3: 2 covenant(s)
Running 3 extraction passes on Facility B (facility_b_credit_agreement.txt)...
  Pass 1: 2 covenant(s)
  Pass 2: 2 covenant(s)
  Pass 3: 2 covenant(s)
Running 3 extraction passes on Facility C (original) (facility_c_credit_agreement.txt)...
  Pass 1: 1 covenant(s)
  Pass 2: 1 covenant(s)
  Pass 3: 1 covenant(s)
Running 3 extraction passes on Facility C (Amendment 1) (facility_c_amendment_1.txt)...
  Pass 1: 1 covenant(s)
  Pass 2: 1 covenant(s)
  Pass 3: 1 covenant(s)

All passes complete. Raw output saved to ../records/02_raw_passes.json


In [16]:
with open(RECORDS_DIR / "02_raw_passes.json") as f:
    all_passes = json.load(f)

print(f"Reloaded raw passes for {len(all_passes)} document(s) from disk. No API calls made.")

Reloaded raw passes for 4 document(s) from disk. No API calls made.


## Schema validation

Before scoring agreement across passes, each individual extracted record is checked against a strict schema using Pydantic. This catches structurally incomplete records — a covenant missing a required field — independently of whether other passes agree with it. One real finding from testing this before handing it over: an empty `threshold_tiers` list technically satisfies Pydantic's type check even though it's practically useless. That's exactly why schema validation alone isn't the whole confidence signal — it's combined with multi-pass agreement next, not used on its own.

In [17]:
from pydantic import BaseModel, ValidationError
from typing import Optional, List


class ThresholdTier(BaseModel):
    applies_from: Optional[str] = None
    applies_to: Optional[str] = None
    threshold_value: str
    citation: str
    citation_verified: Optional[bool] = None


class CovenantRecord(BaseModel):
    covenant_name: str
    definition_text: Optional[str] = None
    definition_citation: Optional[str] = None
    definition_citation_verified: Optional[bool] = None
    threshold_operator: Optional[str] = None
    threshold_tiers: List[ThresholdTier]
    supersedes: Optional[str] = None
    document_effective_date: Optional[str] = None
    extraction_notes: Optional[str] = None


def schema_valid(covenant_dict: dict) -> tuple[bool, str | None]:
    try:
        CovenantRecord(**covenant_dict)
        return True, None
    except ValidationError as e:
        return False, str(e)

## Confidence scoring: aligning covenants across passes

The three passes are independent calls, so nothing guarantees they list covenants in the same order, or even use the exact same name each time — a paraphrase like "Delinquency Ratio" vs. "Delinquency Ratio / Delinquency Trigger" is a real possibility. Matching by name text would be fragile.

Instead, covenants are aligned across passes by the **citation of their threshold** — the exact source span each pass quoted. Citations are meant to be verbatim quotes from a static, unchanging document, so if extraction is genuinely grounded rather than drifting, the same real covenant should produce the same citation across independent passes, even if the covenant's *name* is worded slightly differently each time.

For each distinct citation found across the three passes:
- **Agreement rate** — in how many of the three passes did this exact citation appear?
- **Threshold consistency** — when it did appear, did every pass report the same threshold value for it?
- **Citation verified** — does it hold up against the source text (from Notebook 01's check)?

A covenant is **high confidence** only if all three hold: found in every pass, consistent threshold value, and verified. Anything less — missing from a pass, or a disagreement on the actual number — is flagged **low confidence, needs review**.

In [19]:
def build_confidence_report(passes: list[list[dict]]) -> list[dict]:
    n_passes = len(passes)
    all_occurrences = []
    for pass_idx, covenants in enumerate(passes):
        for covenant in covenants:
            tiers = covenant.get("threshold_tiers", [])
            for tier in tiers:  # every tier, not just the first
                all_occurrences.append((pass_idx, covenant, tier))

    groups = []
    for occ in all_occurrences:
        _, _, tier = occ
        citation = tier["citation"]
        placed = False
        for group in groups:
            rep_citation = group[0][2]["citation"]
            if citation in rep_citation or rep_citation in citation:
                group.append(occ)
                placed = True
                break
        if not placed:
            groups.append([occ])

    report = []
    for group in groups:
        pass_indices = sorted({p for p, _, _ in group})
        agreement_rate = len(pass_indices) / n_passes
        threshold_values = sorted({t["threshold_value"] for _, _, t in group})
        threshold_consistent = len(threshold_values) == 1
        citation_verified = all(t.get("citation_verified", False) for _, _, t in group)

        schema_valid_all = True
        for _, covenant, _ in group:
            valid, _ = schema_valid(covenant)
            if not valid:
                schema_valid_all = False

        longest_citation = max((t["citation"] for _, _, t in group), key=len)

        confidence = (
            "high"
            if (agreement_rate == 1.0 and threshold_consistent and citation_verified and schema_valid_all)
            else "low - needs review"
        )

        report.append({
            "covenant_name": group[0][1]["covenant_name"],
            "primary_citation": longest_citation,
            "seen_in_passes": pass_indices,
            "agreement_rate": agreement_rate,
            "threshold_values_seen": threshold_values,
            "threshold_consistent": threshold_consistent,
            "citation_verified": citation_verified,
            "schema_valid": schema_valid_all,
            "confidence": confidence,
        })

    return report

## What "needs review" means here

This notebook doesn't decide whether a covenant is *correct* — it decides whether a covenant is *trustworthy enough to proceed without a human looking at it first*. High-confidence items pass through automatically. Anything flagged low-confidence goes into a review queue and is not treated as reliable until a person has actually looked at it — nothing covenant-critical moves forward on the strength of the model's word alone.

In [20]:
final_report = {}
review_queue = []

for filename, passes in all_passes.items():
    report = build_confidence_report(passes)
    final_report[filename] = report
    print(f"\n=== {filename} ===")
    for r in report:
        print(f"  {r['covenant_name']}: {r['confidence']}  "
              f"(agreement={r['agreement_rate']:.2f}, thresholds_seen={r['threshold_values_seen']})")
        if r["confidence"] != "high":
            review_queue.append({"source_document": filename, **r})

output_path = RECORDS_DIR / "02_confidence_output.json"
output_path.write_text(json.dumps({
    "confidence_report": final_report,
    "review_queue": review_queue,
}, indent=2))

print(f"\n{len(review_queue)} item(s) flagged for human review.")
print(f"Saved to {output_path}")


=== facility_a_credit_agreement.txt ===
  Delinquency Ratio / Delinquency Trigger: high  (agreement=1.00, thresholds_seen=['5.00%'])
  Overcollateralization Test: high  (agreement=1.00, thresholds_seen=['8.00%'])

=== facility_b_credit_agreement.txt ===
  Delinquency Ratio / Delinquency Trigger: high  (agreement=1.00, thresholds_seen=['4.00%'])
  Reserve Account Deficiency / Required Reserve Amount: low - needs review  (agreement=1.00, thresholds_seen=['2.00%', '2.00% of the Original Pool Balance'])

=== facility_c_credit_agreement.txt ===
  Consolidated Net Leverage Ratio: high  (agreement=1.00, thresholds_seen=['3.50:1.00'])

=== facility_c_amendment_1.txt ===
  Consolidated Net Leverage Ratio: high  (agreement=1.00, thresholds_seen=['3.50:1.00'])
  Consolidated Net Leverage Ratio: high  (agreement=1.00, thresholds_seen=['4.00:1.00'])

1 item(s) flagged for human review.
Saved to ../records/02_confidence_output.json


In [23]:
if review_queue:
    print("Items requiring human review, and why:\n")
    for item in review_queue:
        print(f"{item['source_document']} | {item['covenant_name']}")
        print(f"  Reason: ", end="")
        if not item["threshold_consistent"]:
            print(f"passes disagreed on the value itself: {item['threshold_values_seen']}")
        elif item["agreement_rate"] < 1.0:
            print(f"only found in {item['agreement_rate']:.0%} of passes")
        elif not item["citation_verified"]:
            print("citation could not be verified against source text")
        print()

Items requiring human review, and why:

facility_b_credit_agreement.txt | Reserve Account Deficiency / Required Reserve Amount
  Reason: passes disagreed on the value itself: ['2.00%', '2.00% of the Original Pool Balance']



## Summary

In [24]:
with open(RECORDS_DIR / "02_confidence_output.json") as f:
    saved = json.load(f)

print("=== Final confidence summary (read back from saved file) ===\n")
for filename, report in saved["confidence_report"].items():
    print(f"{filename}:")
    for r in report:
        print(f"  {r['covenant_name']}: {r['confidence']}")

print(f"\nTotal items in review queue: {len(saved['review_queue'])}")
if saved["review_queue"]:
    print("Flagged for review:")
    for item in saved["review_queue"]:
        print(f"  - {item['source_document']}: {item['covenant_name']} ({item['confidence']})")

=== Final confidence summary (read back from saved file) ===

facility_a_credit_agreement.txt:
  Delinquency Ratio / Delinquency Trigger: high
  Overcollateralization Test: high
facility_b_credit_agreement.txt:
  Delinquency Ratio / Delinquency Trigger: high
  Reserve Account Deficiency / Required Reserve Amount: low - needs review
facility_c_credit_agreement.txt:
  Consolidated Net Leverage Ratio: high
facility_c_amendment_1.txt:
  Consolidated Net Leverage Ratio: high
  Consolidated Net Leverage Ratio: high

Total items in review queue: 1
Flagged for review:
  - facility_b_credit_agreement.txt: Reserve Account Deficiency / Required Reserve Amount (low - needs review)
